# KAN Semi-Infinite-Domain Hyperparameter Optimization

In [1]:
import pandas as pd

In [2]:
import os
import sys
from datetime import datetime
from importlib import reload

current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)

import joblib
import optuna
import pandas as pd
import torch
import pinns_infinite
import pinns_semi_infinite
import semi_infinite
from pinns_semi_infinite import run_experiment_semi_inf, set_seed

reload(pinns_infinite)
reload(semi_infinite)
reload(pinns_semi_infinite)
torch.set_default_dtype(torch.float32)
set_seed(42)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Optuna Search Configuration

In [3]:
KAN_SEARCH_SPACE = {
    'hidden_layers': [1, 2, 3],
    'hidden_units': [15, 25, 35],
    'grid_size': [3, 5, 7],
    'spline_order': [2, 3, 4],
    'learning_rate': [1e-4, 1e-3, 1e-2],
}

N_TRIALS = 50
ADAM_ITERS = 2000
LBFGS_ITERS = 2000
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
results_dir = f'results_kan_semi_infinite_optuna_{timestamp}'
os.makedirs(results_dir, exist_ok=True)
print(f'Results will be saved to: {results_dir}')
print(f'Optuna trials: {N_TRIALS}')

Results will be saved to: results_kan_semi_infinite_optuna_2026-09-20_20-46-49
Optuna trials: 50


## Objective Function

In [4]:
def objective(trial):
    config = {
        name: trial.suggest_categorical(name, values)
        for name, values in KAN_SEARCH_SPACE.items()
    }
        
    print(
        f'\n--- Trial {trial.number}: '
        f"L={config['hidden_layers']}, N={config['hidden_units']}, "
        f"grid={config['grid_size']}, order={config['spline_order']}, "
        f"lr={config['learning_rate']:.0e} ---"
    )

    try:
        result = run_experiment_semi_inf(
            model_type='KAN',
            hidden_layers=config['hidden_layers'],
            hidden_units=config['hidden_units'],
            grid_size=config['grid_size'],
            spline_order=config['spline_order'],
            adam_lr=config['learning_rate'],
            device=device,
            adam_iters=ADAM_ITERS,
            lbfgs_iters=LBFGS_ITERS,
            results_dir=results_dir,
            eval_domain=(-10.0, 10.0, -10.0, 0.0),
        )
    except Exception as error:
        print(f'Trial {trial.number} failed: {error}')
        raise optuna.exceptions.TrialPruned() from error

    err_u = float(result['err_u_global'])
    err_k = float(result['err_k_global'])
    compute_time = float(result['compute_time_sec'])
    mean_global_error = 0.5 * (err_u + err_k)
    trial.set_user_attr('err_u', err_u)
    trial.set_user_attr('err_k', err_k)
    trial.set_user_attr('compute_time_sec', compute_time)
    print(
        f'Success! Time: {compute_time:.2f}s | '
        f'Err U: {err_u:.3e} | Err K: {err_k:.3e} | '
        f'Mean error: {mean_global_error:.3e}'
    )
    return mean_global_error

## Run Optimization

In [5]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    study_name=f'kan_semi_infinite_domain_{timestamp}',
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    catch=(RuntimeError, ValueError),
)

print('\n========================================')
print('BEST KAN SEMI-INFINITE CONFIGURATION')
print('========================================')
print(f'Mean global error: {study.best_value:.6e}')
print('Parameters:')
for name, value in study.best_params.items():
    print(f'  {name}: {value}')

[I 2026-09-20 20:46:52,136] A new study created in memory with name: kan_semi_infinite_domain_2026-09-20_20-46-49



--- Trial 0: L=2, N=15, grid=5, order=4, lr=1e-04 ---


/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[I 2026-09-20 20:52:41,829] Trial 0 finished with value: 0.0017081124999083958 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'grid_size': 5, 'spline_order': 4, 'learning_rate': 0.0001}. Best is trial 0 with value: 0.0017081124999083958.



[KAN] L=2, N=15 | Params: 5,940 | Mean Err: 1.708e-03 | Saved to 'results_kan_semi_infinite_optuna_2026-09-20_20-46-49/KAN/2026-09-20_20-46-52/'.
Success! Time: 349.68s | Err U: 2.656e-03 | Err K: 7.601e-04 | Mean error: 1.708e-03

--- Trial 1: L=3, N=35, grid=7, order=3, lr=1e-03 ---


[I 2026-09-20 20:59:15,979] Trial 1 finished with value: 0.0018595801900212735 and parameters: {'hidden_layers': 3, 'hidden_units': 35, 'grid_size': 7, 'spline_order': 3, 'learning_rate': 0.001}. Best is trial 0 with value: 0.0017081124999083958.


...


[KAN] L=3, N=15 | Params: 9,900 | Mean Err: 1.860e-03 | Saved to 'results_kan_semi_infinite_optuna_2026-09-20_20-46-49/KAN/2026-09-21_01-17-25/'.
Success! Time: 386.46s | Err U: 2.962e-03 | Err K: 7.583e-04 | Mean error: 1.860e-03

--- Trial 48: L=3, N=15, grid=5, order=3, lr=1e-02 ---


[I 2026-09-21 01:30:20,638] Trial 48 finished with value: 0.0011665818849430018 and parameters: {'hidden_layers': 3, 'hidden_units': 15, 'grid_size': 5, 'spline_order': 3, 'learning_rate': 0.01}. Best is trial 46 with value: 0.0008376314407819692.



[KAN] L=3, N=15 | Params: 9,900 | Mean Err: 1.167e-03 | Saved to 'results_kan_semi_infinite_optuna_2026-09-20_20-46-49/KAN/2026-09-21_01-23-51/'.
Success! Time: 388.74s | Err U: 1.825e-03 | Err K: 5.083e-04 | Mean error: 1.167e-03

--- Trial 49: L=3, N=25, grid=3, order=4, lr=1e-02 ---


[I 2026-09-21 01:38:36,989] Trial 49 finished with value: 0.0013479645757251504 and parameters: {'hidden_layers': 3, 'hidden_units': 25, 'grid_size': 3, 'spline_order': 4, 'learning_rate': 0.01}. Best is trial 46 with value: 0.0008376314407819692.



[KAN] L=3, N=25 | Params: 23,850 | Mean Err: 1.348e-03 | Saved to 'results_kan_semi_infinite_optuna_2026-09-20_20-46-49/KAN/2026-09-21_01-30-20/'.
Success! Time: 496.33s | Err U: 1.665e-03 | Err K: 1.030e-03 | Mean error: 1.348e-03

BEST KAN SEMI-INFINITE CONFIGURATION
Mean global error: 8.376314e-04
Parameters:
  hidden_layers: 3
  hidden_units: 15
  grid_size: 5
  spline_order: 3
  learning_rate: 0.01


In [6]:
data_dir = os.path.join(results_dir, 'data')
os.makedirs(data_dir, exist_ok=True)
joblib.dump(study, os.path.join(data_dir, 'study.pkl'))
joblib.dump(study, os.path.join(data_dir, f'study_{timestamp}.pkl'))
study_df = study.trials_dataframe()
study_csv_path = os.path.join(data_dir, 'study.csv')
study_df.to_csv(study_csv_path, index=False)
filtered_df = study_df[
    (study_df['params_hidden_layers'] == 3)
    & (study_df['params_hidden_units'] == 25)
].sort_values(by='value', ascending=True)
filtered_csv_path = os.path.join(data_dir, 'study_filtered_sorted.csv')
filtered_df.to_csv(filtered_csv_path, index=False)
print(f'Saved study to: {data_dir}')
print(f'Saved trial summary to: {study_csv_path}')
print(f'Saved filtered summary to: {filtered_csv_path}')

Saved study to: results_kan_semi_infinite_optuna_2026-09-20_20-46-49/data
Saved trial summary to: results_kan_semi_infinite_optuna_2026-09-20_20-46-49/data/study.csv
Saved filtered summary to: results_kan_semi_infinite_optuna_2026-09-20_20-46-49/data/study_filtered_sorted.csv
